*Notebook Last Validated: pending*

# Staggered Federated Learning with PyTorch Lightning (Chemprop) on Rhino FCP

This notebook runs the `lightning-chemprop` example on Rhino's Federated Computing Platform (FCP): N sites that share one GPU agent train a Chemprop model, taking turns on the GPU (at most 4 at a time). It makes the data shards, creates the data schema, registers one dataset per site, creates the Code Object and runs the training.

See `README.md` in this folder for what the example does and how to set it up, and `../README.md` for the staggered FL guide.

#### Before you run this notebook
1. `pip install rhino_health pandas` (rhino_health 2.1.25 or newer, for `CodeTypes.NVIDIA_FLARE_V2_8`)
2. A project with N collaborating workgroups whose sites run on the same GPU agent (README, "Running this example on FCP", step 1)
3. The image built and pushed (README step 2)

## Setup

In [ ]:
import json
import math
import os
from getpass import getpass

import rhino_health as rh
from rhino_health.lib.endpoints.code_object.code_object_dataclass import (
    CodeObjectCreateInput,
    CodeTypes,
    ModelTrainInput,
)

# Pin the working directory to this notebook's folder so relative paths resolve
# regardless of where the notebook is opened from (e.g. VS Code, Jupyter Lab, etc.)
os.chdir(os.path.dirname(os.path.abspath(globals().get('__vsc_ipynb_file__', 'notebook.ipynb'))))
print("Setup Complete")

## Container image

Paste the URI that `docker-push.sh` printed (README step 2).

In [ ]:
CONTAINER_IMAGE_URI = "<YOUR_CONTAINER_IMAGE_URI>"  # UPDATE THIS TO THE URI PRINTED BY docker-push.sh

if CONTAINER_IMAGE_URI.startswith("<"):
    raise ValueError("Set CONTAINER_IMAGE_URI to the URI printed by docker-push.sh before continuing.")
print("Completed.")

## Authentication

Log in to FCP; you'll be prompted for your password.

**Variables to adjust:**
- `USERNAME` - your Rhino FCP username (typically your email)

In [ ]:
USERNAME = "<YOUR_USERNAME>"  # REPLACE WITH YOUR RHINO FCP USERNAME

session = rh.login(username=USERNAME, password=getpass())
print("Logged In")

## Project and site workgroups

FCP runs one federated learning site per dataset, and each workgroup holds one dataset here. This finds your project and its collaborating workgroups whose name starts with `SITE_WORKGROUP_PREFIX`: these are the N sites.

**Variables to adjust:**
- `PROJECT_NAME` - the project your workgroups collaborate in
- `SITE_WORKGROUP_PREFIX` - the common prefix of the site workgroups' names (e.g. `stagger-site-` for `stagger-site-01` ... `stagger-site-08`)

In [ ]:
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
SITE_WORKGROUP_PREFIX = "<YOUR_SITE_WORKGROUP_PREFIX>"

project = session.project.get_project_by_name(PROJECT_NAME)
if project is None:
    raise ValueError(f"Project {PROJECT_NAME!r} not found.")
site_workgroups = sorted(
    (workgroup for workgroup in project.collaborating_workgroups if workgroup.name.startswith(SITE_WORKGROUP_PREFIX)),
    key=lambda workgroup: workgroup.name,
)
if not site_workgroups:
    raise ValueError(f"No collaborating workgroups starting with {SITE_WORKGROUP_PREFIX!r} in project {PROJECT_NAME!r}.")
for workgroup in site_workgroups:
    print(f"{workgroup.name} ({workgroup.uid})")
print(f"{len(site_workgroups)} sites")

## Data shards

Splits the CYP3A4 data in this repository into one CSV per site: `data/shards/shard_01.csv` ... `shard_NN.csv`, with the columns `id`, `smiles` and `cyp3a4`.

**Then copy the `data/shards` folder to your GPU agent's storage** (the on-prem path your agent imports datasets from). The notebook can't do this step: the files must exist on the agent before the datasets are registered below.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "data/make_shards.py", "--num-shards", str(len(site_workgroups))], check=True
)

## Data schema

Creates the data schema for the shards (columns `id`, `smiles`, `cyp3a4`) in your project, owned by the project's primary workgroup. If a schema with this name already exists in the project, it is reused.

In [ ]:
from rhino_health.lib.endpoints.data_schema.data_schema_dataclass import DataSchemaCreateInput

DATA_SCHEMA_NAME = "CYP3A4 Inhibition - Staggered FL"
DATA_SCHEMA_FIELDS = [
    "Variable Name,id,smiles,cyp3a4\n",
    "Identifier,id,smiles,cyp3a4\n",
    "Description,Molecule id,Molecule SMILES string,CYP3A4 inhibitor (1) or not (0)\n",
    "Role,Input,Input,Output\n",
    "Type,String,String,Integer\n",
    "Type Parameters,,,\n",
    "Units,,,\n",
    "May Contain PHI?,FALSE,FALSE,FALSE\n",
    "Permissions,Default,Default,Default\n",
]

data_schema = session.project.get_data_schema_by_name(DATA_SCHEMA_NAME, project_uid=project.uid)
if data_schema is None:
    data_schema = session.data_schema.create_data_schema(
        DataSchemaCreateInput(
            name=DATA_SCHEMA_NAME,
            description="Molecule SMILES and whether the molecule inhibits CYP3A4",
            primary_workgroup_uid=project.primary_workgroup_uid,
            project_uid=project.uid,
            schema_fields=DATA_SCHEMA_FIELDS,
        )
    )
print(f"Data schema: {data_schema.name} ({data_schema.uid})")

## Datasets

Registers shard *i* as a dataset in the *i*-th site workgroup, acting as that workgroup, so every site gets its own share of the data. Datasets that already exist are reused, so the cell is safe to re-run. It then lists every dataset in the project whose name starts with `DATASET_NAME_PREFIX`: one per site.

Registering as a workgroup needs you to be a member of it. If you are not a member of every site workgroup, set `REGISTER_DATASETS = False` and register the shards in the FCP UI instead, once per workgroup (**Datasets → Add Dataset**, the data schema above, the names `cyp3a4-shard-01` ... in workgroup order).

**Variables to adjust:**
- `SHARDS_LOCATION` - the path on the GPU agent where you copied `data/shards`
- `DATASET_NAME_PREFIX` - the common prefix of the shard datasets
- `REGISTER_DATASETS` - `False` to only list datasets you registered in the UI

In [ ]:
from rhino_health.lib.endpoints.dataset.dataset_dataclass import DatasetCreateInput

SHARDS_LOCATION = "<PATH_TO_SHARDS_ON_THE_AGENT>"
DATASET_NAME_PREFIX = "cyp3a4-shard-"
REGISTER_DATASETS = True

if REGISTER_DATASETS:
    for index, workgroup in enumerate(site_workgroups, start=1):
        session.switch_workgroup(workgroup)
        try:
            dataset = session.dataset.add_dataset(
                DatasetCreateInput(
                    name=f"{DATASET_NAME_PREFIX}{index:02d}",
                    description=f"CYP3A4 shard {index} of {len(site_workgroups)}",
                    project_uid=project.uid,
                    workgroup_uid=workgroup.uid,
                    data_schema_uid=data_schema.uid,
                    csv_filesystem_location=f"{SHARDS_LOCATION}/shard_{index:02d}.csv",
                    method="filesystem",
                    is_data_deidentified=True,
                )
            )
        finally:
            session.switch_workgroup(session.current_user.primary_workgroup)
        print(f"{workgroup.name}: {dataset.name} ({dataset.import_status})")
        if dataset.import_status == "Error":
            raise RuntimeError(f"Importing {dataset.name} failed: check that {SHARDS_LOCATION} exists on the agent.")

datasets = sorted(
    (dataset for dataset in session.project.get_datasets(project.uid) if dataset.name.startswith(DATASET_NAME_PREFIX)),
    key=lambda dataset: dataset.name,
)
if len(datasets) != len(site_workgroups):
    raise ValueError(f"Found {len(datasets)} datasets starting with {DATASET_NAME_PREFIX!r}, expected one per site ({len(site_workgroups)}).")
for dataset in datasets:
    print(f"{dataset.name} ({dataset.uid})")

## NVFlare Code Object Creation

The Code Object points at the image you pushed. `return_existing=False, add_version_if_exists=True` makes a new version when one with this name exists, so a new image URI takes effect on re-runs.

In [ ]:
code_object_input = CodeObjectCreateInput(
    name="Staggered FL - Lightning Chemprop",
    description="Chemprop CYP3A4 classifier, sites take turns on a shared GPU",
    input_data_schema_uids=[datasets[0].data_schema_uid],
    output_data_schema_uids=[None],
    project_uid=project.uid,
    code_type=CodeTypes.NVIDIA_FLARE_V2_8,  # must match requirements.txt: nvflare==2.8.1
    config={"container_image_uri": CONTAINER_IMAGE_URI},
)
code_object = session.code_object.create_code_object(
    code_object_input, return_existing=False, add_version_if_exists=True
)
code_object = code_object.wait_for_build()  # returns immediately - the image is already built
print(f"Created code object: {code_object.name} ({code_object.uid})")

## Job configuration

Reads the two configs, removes their `//` comment lines (JSON doesn't allow comments), and fits them to your datasets:
- `min_clients` is set to the number of datasets: FCP starts the job only when every site is up.
- `min_successful_clients` (sites that must succeed in each round) is kept at most `min_clients`.
- `timeout_seconds`, after which FCP stops the whole run, is computed from the rounds, the sites and K, because sites take turns.

**Variables to adjust:**
- `LOCAL_TRAINING_SECONDS` - how long one site trains per round when K sites train at once (~75 s for this example on an NVIDIA L4)
- `OVERHEAD_SECONDS` - time per turn outside training (~10 s with one process per site)

In [ ]:
LOCAL_TRAINING_SECONDS = 75
OVERHEAD_SECONDS = 10
SAFETY_FACTOR = 1.3


def load_config(path):
    """Read a commented example config and return it as a dict."""
    with open(path) as f:
        return json.loads("".join(line for line in f if not line.lstrip().startswith("//")))


config_fed_client = load_config("app/config/config_fed_client.json")
config_fed_server = load_config("app/config/config_fed_server.json")

workflow_args = config_fed_server["workflows"][0]["args"]
num_sites = len(datasets)
workflow_args["min_clients"] = num_sites
workflow_args["min_successful_clients"] = min(workflow_args.get("min_successful_clients", num_sites), num_sites)

waves_per_round = math.ceil(num_sites / workflow_args["max_concurrent"])
timeout_seconds = math.ceil(
    workflow_args["num_rounds"] * waves_per_round * (LOCAL_TRAINING_SECONDS + OVERHEAD_SECONDS) * SAFETY_FACTOR
)
timeout_seconds = max(timeout_seconds, 1800)  # leaves room for start-up on small jobs
print(
    f"{num_sites} sites, K = {workflow_args['max_concurrent']} ({waves_per_round} waves per round), "
    f"{workflow_args['num_rounds']} rounds, min_successful_clients = {workflow_args['min_successful_clients']}, "
    f"timeout_seconds = {timeout_seconds}"
)

## Federated training run

Starts the run on every dataset (one site each) and waits for it to finish.

In [ ]:
run_params = ModelTrainInput(
    code_object_uid=code_object.uid,
    input_dataset_uids=[dataset.uid for dataset in datasets],
    one_fl_client_per_dataset=True,
    validation_dataset_uids=[],
    validation_datasets_inference_suffix="",
    timeout_seconds=timeout_seconds,
    config_fed_client=json.dumps(config_fed_client),
    config_fed_server=json.dumps(config_fed_server),
)

model_train = session.code_object.train_model(run_params)
code_run = model_train.wait_for_completion(timeout_seconds, poll_frequency=30)
print(f"Training finished with status: {code_run.status}")

In [ ]:
# If the run failed, its errors are here. The server log (FCP UI, the run's logs) has a "Site ... failed" or
# "Job stopped ..." line explaining which sites failed and why; see "Failures" in ../README.md.
print(code_run.errors)

## Check the staggering

Download the run's server log from the FCP UI and run, from this folder:
```bash
python ../stagger_report.py <server-log-file>
```
It prints one line per round (duration, failed sites, the most sites that trained at once per agent) and ends with `OK` when no agent went over K.

## Model Parameters Download

Downloads the trained global model. You can also download it from the FCP UI: Code Runs, the 3 dots next to the run, "Download Model Parameters".

In [ ]:
weights = session.code_run.get_model_params(code_run.uid)
with open("model_parameters.pt", "wb") as f:
    f.write(weights.getbuffer())
print("Saved model to model_parameters.pt")

## Cleanup

Removes the Code Object this notebook created. Your project, workgroups, data schema and datasets are left as they are, and so are the shards on the agent and the pushed image.

In [ ]:
CLEANUP = False  # Set to True to delete the Code Object created above

if CLEANUP:
    session.code_object.remove_code_object(code_object)
    print("Removed the code object from FCP.")
else:
    print("Skipped - set CLEANUP = True above to remove the Code Object this notebook created.")

## Additional Resources

- [Staggered FL guide](../README.md)
- [Rhino SDK Documentation](https://rhinohealth.github.io/rhino_sdk_docs/html/autoapi/index.html)
- [Rhino FCP Platform Documentation](https://docs.rhinofcp.com/)
- [NVFlare GitHub](https://github.com/NVIDIA/NVFlare/tree/main)